In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


In [ ]:
import os
os.chdir(r'C:\Users\thoma\OneDrive - stevens.edu\Documents\GitHub\CS513_Project')

In [ ]:
df = pd.read_csv('test.csv', na_values='?')
df.drop(columns=['Unnamed: 0'], inplace=True)
df = df.dropna()

In [ ]:
df.head()

In [ ]:
#Feature engineering

#Total delay rather than 
df['Total Delay'] = df['Departure Delay in Minutes'] + df['Arrival Delay in Minutes']

#Turns delay into a categorical feature 
def bucket_total_delay(minutes):
    if minutes <= 15:
        return 'On Time'
    elif minutes <= 60:
        return 'Small Delay'
    else:
        return 'Big Delay'

# Apply the function to create a new feature
df['Delay Category'] = df['Total Delay'].apply(bucket_total_delay)

def bucket_rating(x):
    if x <= 2:
        return 'Low'
    elif x == 3:
        return 'Medium'
    else:
        return 'High'

rating_columns = [ 'Inflight wifi service','Food and drink', 'Seat comfort',
                  'Inflight entertainment', 'On-board service', 'Leg room service',
                  'Baggage handling', 'Checkin service', 'Cleanliness', 'Online boarding']

for col in rating_columns:
    df[col] = df[col].apply(bucket_rating)




In [ ]:
attr = df.drop(columns=['satisfaction','id'])
target = df['satisfaction']

# Test/Train split
attr_train, attr_test, target_train, target_test = train_test_split(attr, target, test_size=0.2, random_state=5)
categorical_columns = [
    'Gender',
    'Customer Type',
    'Type of Travel',
    'Class',
    'Inflight wifi service',
    'Food and drink',
    'Seat comfort',
    'Inflight entertainment',
    'On-board service',
    'Leg room service',
    'Baggage handling',
    'Checkin service',
    'Cleanliness',
    'Online boarding',
    'Delay Category'
]
# Create preprocessor for categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), categorical_columns) ])
# Create Naive Bayes model
cnb = CategoricalNB()

# Create a pipeline with preprocessing and model
model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('classifier', cnb)])
#Perform grid search and smoothing
param_grid = {
    'classifier__alpha': [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
}
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(attr_train, target_train)
print("Best alpha:", grid_search.best_params_)
print("Best cross-validation accuracy:", grid_search.best_score_)


# Train the model
model.fit(attr_train, target_train)

# # Make predictions on the test set
best_model = grid_search.best_estimator_
target_pred = best_model.predict(attr_test)

In [ ]:
attr_test.head()

In [ ]:

# Evaluate the accuracy
accuracy = accuracy_score(target_test, target_pred)
print(f'Accuracy: {accuracy:.2f}\n')

# Compute the confusion matrix
cm = confusion_matrix(target_test, target_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Unsatisfied', 'Satisfied'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Naive Bayes ? Confusion Matrix')
plt.grid(False)
plt.show()

# Print classification report
print('\nClassification Report:')
print(classification_report(target_test, target_pred, target_names=['Unsatisfied', 'Satisfied']))
